# ML_NOTEBOOK3 — Dengue Early Warning for Bangladesh
## Calibrated forecasting and the resolution-dependence of forecast skill, 2019–2026

Target venue: **7th IEEE ICEEICT 2027**, MIST Dhaka.
Supersedes `ml_notebook2.py`. Every defect found in the 4 Sep review is fixed here,
and the additions the plan required (resolution ablation, bootstrap intervals,
covariate ablation) plus the higher-value ideas (probabilistic baseline, WIS,
conditional coverage, adaptive conformal, forward test, lead time, Diebold–Mariano)
are implemented.

### Required Kaggle inputs
| Dataset | Files used |
|---|---|
| 64-district weekly panel (figshare 10.6084/m9.figshare.33040637) | `Dengue.csv` |
| Divisional daily bulletin | any CSV with `date, division, Patients, max temp, min temp, rainfall, humidity` |

### Accelerator
**Use CPU, not T4 ×2.** Every model here is a gradient-boosted tree on ≤16k rows;
LightGBM's GPU path is slower than CPU at this size and adds a build dependency.
The notebook runs end to end in roughly 6–10 minutes on Kaggle's 4-core CPU.


In [ ]:
# =============================================================================
# CELL 0 — Environment, reproducibility, paths
# =============================================================================
import os, sys, glob, json, random, logging, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)

IS_KAGGLE = os.path.exists("/kaggle")
BASE_INPUT = "/kaggle/input" if IS_KAGGLE else os.path.abspath("./data")
WORKDIR    = "/kaggle/working" if IS_KAGGLE else os.path.abspath("./artifacts")
RESULTS    = os.path.join(WORKDIR, "results")
FIGURES    = os.path.join(WORKDIR, "figures")
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger("iceeict")

import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, roc_auc_score,
                             average_precision_score, precision_recall_curve)
from scipy import stats

# ---- Analysis constants. Declared once; used everywhere. --------------------
HORIZONS        = [1, 2, 3, 4]
TEST_YEARS      = [2024, 2025]      # 2026 excluded here; used as a forward test in Cell 12
CAL_WEEKS       = 26                # DEFAULT ONLY - selected empirically in Cell 4b
ALARM_QUANTILE  = 0.80              # per-unit outbreak threshold
ALARM_MIN_CASES = 5
ALARM_TRAIN_MAX = 2023              # threshold derived from <= this year only
NOMINAL         = 0.90

# DEFAULTS ONLY. Cell 4b replaces these with values chosen on an inner validation
# split that never sees a test year. Nothing downstream reads them before that.
LGB_REG = dict(n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=31,
               subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
               random_state=SEED, verbose=-1, n_jobs=-1)
TWEEDIE_P       = 1.4               # selected in Cell 4b
GROWTH_OBJ_NAME = "L1"              # selected in Cell 4b
ALPHA           = 0.05              # significance level for every test in this notebook
LGB_CLF = dict(n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=31,
               subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
               random_state=SEED, verbose=-1, n_jobs=-1)

# Okabe-Ito, ordered so every adjacent pair clears CVD ΔE >= 8 (validated).
PAL = {"blue": "#0072B2", "verm": "#D55E00", "green": "#009E73",
       "orange": "#E69F00", "purple": "#CC79A7", "grey": "#4D4D4D"}

log.info("=" * 78)
log.info("ML_NOTEBOOK3 · ICEEICT 2027 · seed=%d · %s", SEED, "KAGGLE" if IS_KAGGLE else "LOCAL")
log.info("python %s · pandas %s · numpy %s · lightgbm %s",
         sys.version.split()[0], pd.__version__, np.__version__, lgb.__version__)
log.info("results -> %s", RESULTS)
log.info("=" * 78)

_TABLES = {}
def save_table(name, df, caption=""):
    """Persist a results table as CSV and register it in the manifest."""
    p = os.path.join(RESULTS, f"{name}.csv")
    df.to_csv(p, index=False)
    _TABLES[name] = {"rows": len(df), "caption": caption, "path": p}
    log.info("saved %-42s %3d rows", name + ".csv", len(df))
    return df


In [ ]:
# =============================================================================
# CELL 1 — Panel loaders. Both resolutions produce the SAME canonical schema.
# =============================================================================
# canonical columns: unit | block | year | week_start | epi_week | cases | covariates
# `unit`  = the modelling unit (district, or division at coarse resolution)
# `block` = the spatial holdout group (division at both resolutions)

def _find(*names, root=BASE_INPUT):
    """Locate the first matching file anywhere under root (case-insensitive)."""
    for n in names:
        hits = glob.glob(os.path.join(root, "**", n), recursive=True)
        if hits:
            return sorted(hits)[0]
    low = {n.lower() for n in names}
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.lower() in low:
                return os.path.join(r, f)
    return None


def load_district_panel():
    p = _find("Dengue.csv", "district_week_panel.csv")
    if p is None:
        raise FileNotFoundError(
            "District panel not found. Attach the 64-district weekly panel "
            "(figshare 10.6084/m9.figshare.33040637) so that Dengue.csv is under /kaggle/input.")
    log.info("district panel  <- %s", p)
    d = pd.read_csv(p)
    d.columns = [c.strip().lower().replace(" ", "_") for c in d.columns]
    d = d.rename(columns={"district": "unit", "division": "block",
                          "new_cases": "cases", "iso_week": "epi_week"})
    d["week_start"] = pd.to_datetime(d["week_start"], errors="coerce")
    d["cases"] = pd.to_numeric(d["cases"], errors="coerce").fillna(0).clip(lower=0)
    # Log-space transforms of the panel's own spillover columns (contemporaneous, legitimate).
    for c in ["nbr_cases", "nbr_cases_lag1", "nbr_cases_lag2",
              "upstream_pressure", "upstream_pressure_lag1", "upstream_pressure_lag2"]:
        if c in d.columns:
            d["log_" + c] = np.log1p(pd.to_numeric(d[c], errors="coerce").clip(lower=0))
    return d.sort_values(["unit", "week_start"]).reset_index(drop=True)


def load_divisional_panel():
    """Aggregate the daily divisional bulletin to weeks ending Saturday."""
    p = _find("divisional_daily_2022_2025.csv", "Dataset (1).csv", "dataset (1).csv")
    if p is None:                                    # last resort: sniff for the schema
        for cand in glob.glob(os.path.join(BASE_INPUT, "**", "*.csv"), recursive=True):
            try:
                cols = {c.strip().lower() for c in pd.read_csv(cand, nrows=2).columns}
            except Exception:
                continue
            if {"date", "division", "patients"} <= cols:
                p = cand; break
    if p is None:
        log.warning("divisional daily file not found — the resolution ablation will be skipped")
        return None
    log.info("divisional panel <- %s", p)
    d = pd.read_csv(p)
    d.columns = [c.strip().lower() for c in d.columns]
    d["date"] = pd.to_datetime(d["date"], format="mixed", errors="coerce")
    d = d.dropna(subset=["date"])
    d["yw"] = d["date"].dt.to_period("W-SAT")
    w = (d.groupby(["division", "yw"])
           .agg(cases=("patients", "sum"), temp_max=("max temp", "mean"),
                temp_min=("min temp", "mean"), rain=("rainfall", "sum"),
                humidity=("humidity", "mean"))
           .reset_index())
    w["week_start"] = w["yw"].dt.start_time
    w["temp_mean"] = (w["temp_max"] + w["temp_min"]) / 2.0

    # The district panel's climate is NASA POWER + CHIRPS. The daily bulletin carries
    # station-style readings instead, which would confound spatial resolution with
    # climate source in the resolution ablation. If the NASA POWER harvest for the
    # eight division centroids is attached, use it and keep the two panels comparable.
    npw = _find("nasa_power_divisions_daily.csv")
    if npw:
        log.info("divisional climate <- NASA POWER (%s)", os.path.basename(npw))
        n = pd.read_csv(npw)
        n["date"] = pd.to_datetime(n["date"], errors="coerce")
        n["yw"] = n["date"].dt.to_period("W-SAT")
        nw = (n.groupby(["division", "yw"])
                .agg(temp_mean=("temp_mean", "mean"), temp_min=("temp_min", "min"),
                     temp_max=("temp_max", "max"), rain=("rain", "sum"),
                     humidity=("humidity", "mean"))
                .reset_index())
        w = (w.drop(columns=["temp_mean", "temp_min", "temp_max", "rain", "humidity"])
               .merge(nw, on=["division", "yw"], how="left"))
        w["climate_source"] = "NASA POWER"
    else:
        log.warning("nasa_power_divisions_daily.csv not attached — the divisional arm will "
                    "use station climate, which confounds resolution with climate source")
        w["climate_source"] = "daily bulletin (station)"
    w["unit"] = w["division"]; w["block"] = w["division"]
    w["year"] = w["week_start"].dt.year
    w["epi_week"] = w["week_start"].dt.isocalendar().week.astype(int)
    w["cases"] = w["cases"].clip(lower=0)
    # Weekly buckets straddle 1 January, so reconcile against the DAILY source instead
    # of the roll-up; otherwise a few days of January land in the previous year.
    w.attrs["daily_annual"] = d.groupby(d["date"].dt.year)["patients"].sum().to_dict()
    return w.drop(columns=["yw", "division"]).sort_values(["unit", "week_start"]).reset_index(drop=True)


PANEL_D = load_district_panel()
PANEL_V = load_divisional_panel()
log.info("district: %s rows, %d units, %s..%s", f"{len(PANEL_D):,}", PANEL_D.unit.nunique(),
         PANEL_D.week_start.min().date(), PANEL_D.week_start.max().date())
if PANEL_V is not None:
    log.info("division: %s rows, %d units, %s..%s", f"{len(PANEL_V):,}", PANEL_V.unit.nunique(),
             PANEL_V.week_start.min().date(), PANEL_V.week_start.max().date())


In [ ]:
# =============================================================================
# CELL 2 — TABLE 1: surveillance reconciliation against published DGHS totals
# =============================================================================
OFFICIAL_DGHS = {2019: 101354, 2022: 62382, 2023: 321179, 2024: 101214}
# Reconciliation tolerance. The district panel is a curated deposit reconciled by its
# author against the DGHS bulletins; the divisional file is a Kaggle mirror of the same
# bulletins and is known to run ~1.9% light in 2022, so it gets a looser bound.
RECON_TOL_PCT = {"district": 1.0, "divisional": 2.5}

def reconcile(panel, tag):
    rows = []
    annual = panel.attrs.get("daily_annual") or panel.groupby("year")["cases"].sum().to_dict()
    for y, obs in sorted(annual.items()):
        off = OFFICIAL_DGHS.get(int(y))
        d = (obs - off) / off * 100 if off else np.nan
        # 2019 is a partial season in the district panel (rows begin 26 Aug), so a
        # full-year match is not expected and the deposit's README overstates it.
        first, last = panel.week_start.min(), panel.week_start.max()
        partial = (int(y) == 2019) or (int(y) >= 2025)                   or first > pd.Timestamp(f"{int(y)}-01-08")                   or last < pd.Timestamp(f"{int(y)}-12-24")
        rows.append({"panel": tag, "year": int(y), "observed": int(obs),
                     "official_dghs": off if off else np.nan,
                     "delta_pct": round(d, 2) if off else np.nan,
                     "abs_delta_pct": abs(d) if off else np.nan,
                     "tolerance_pct": RECON_TOL_PCT.get(tag, 1.0),
                     "status": "partial season" if partial else
                               ("reconciled" if abs(d) < RECON_TOL_PCT.get(tag, 1.0)
                                else "DISCREPANCY")})
    return pd.DataFrame(rows)

recon = reconcile(PANEL_D, "district")
if PANEL_V is not None:
    recon = pd.concat([recon, reconcile(PANEL_V, "divisional")], ignore_index=True)
save_table("table1_reconciliation", recon, "DGHS surveillance reconciliation")
print(recon.to_string(index=False))

# ---- Data-quality audit. Runs on every execution so the paper's Data section
# ---- can be written from the run output rather than from assertion.
def data_quality_audit(d):
    r = []
    def chk(name, value, status, note=""):
        r.append({"check": name, "value": value, "status": status, "note": note})

    grid = d["unit"].nunique() * d["week_start"].nunique()
    chk("panel completeness", f"{len(d):,} of {grid:,} cells",
        "PASS" if len(d) == grid else "FLAG", "64 units x 254 ISO weeks, Monday-start")
    chk("duplicate unit-weeks", int(d.duplicated(["unit", "week_start"]).sum()), "PASS", "")
    chk("negative or fractional cases",
        int((d.cases < 0).sum() + (d.cases % 1 != 0).sum()), "PASS", "")

    # Zeros that are probably missing reports rather than true zeros.
    gap = (d.cases == 0) & (d.get("currently_admitted", pd.Series(0, index=d.index)) > 0)
    chk("zero-case weeks", f"{100*(d.cases == 0).mean():.1f}%", "INFO", "")
    chk("zeros that look like missing reports", f"{int(gap.sum()):,} rows "
        f"({100*gap.sum()/max((d.cases == 0).sum(),1):.1f}% of zeros)", "FLAG",
        "cases=0 while patients are still admitted; treat zero-inflation claims with care")

    if "new_deaths" in d:
        chk("weeks with deaths but no new cases", int((d.new_deaths > d.cases).sum()), "FLAG",
            "reporting lag: deaths of patients admitted earlier")
    if "cumulative_cases" in d:
        dec = sum(int((g.sort_values("week_start").cumulative_cases.diff().dropna() < 0).any())
                  for _, g in d.groupby(["unit", "year"]))
        chk("seasons where cumulative_cases decreases", f"{dec} of {d.groupby(['unit','year']).ngroups}",
            "FLAG", "one per season at the ISO-year boundary; column is excluded from features")

    for c, lo, hi in [("temp_mean", 5, 42), ("humidity", 10, 100), ("rain", 0, 1500),
                      ("ndvi", -1, 1), ("ndwi", -1, 1)]:
        if c in d:
            chk(f"range check: {c}", f"[{d[c].min():.2f}, {d[c].max():.2f}]",
                "PASS" if ((d[c] >= lo) & (d[c] <= hi)).all() else "FLAG", f"expected [{lo}, {hi}]")
    if {"temp_min", "temp_mean", "temp_max"} <= set(d.columns):
        ok = ((d.temp_min <= d.temp_mean) & (d.temp_mean <= d.temp_max)).all()
        chk("temp_min <= mean <= max", "holds" if ok else "violated", "PASS" if ok else "FLAG", "")

    if "gtrends_dengue" in d:
        per_wk = d.groupby("week_start").gtrends_dengue.nunique().median()
        chk("gtrends_dengue distinct values per week", f"{per_wk:.0f} across {d.unit.nunique()} units",
            "FLAG", "effectively a national seasonal clock; describe it as such, not as district search behaviour")

    nz = d.loc[d.cases > 0, "cases"].astype(int).astype(str).str[0].value_counts(normalize=True)
    benford = pd.Series({str(i): np.log10(1 + 1 / i) for i in range(1, 10)})
    dev = float((nz - benford).abs().max())
    chk("Benford deviation, nonzero counts", f"{dev:.3f}",
        "PASS" if dev < 0.05 else "FLAG", "large deviations would suggest synthetic counts")
    return pd.DataFrame(r)

dq = save_table("table1b_data_quality", data_quality_audit(PANEL_D),
                "Data-quality audit of the district panel")
print(dq.to_string(index=False))


# ---- External cross-checks. The internal audit above cannot detect a dataset that
# ---- is internally consistent but wrong. These two compare the panel against
# ---- sources it was NOT derived from.
def _isokey(s):
    ic = pd.to_datetime(s).dt.isocalendar()
    return list(zip(ic.year.astype(int), ic.week.astype(int)))


def external_cross_checks(panel):
    """Panel cases vs an independently compiled bulletin; panel climate vs NASA POWER.

    Everything is aligned on the ISO (year, week) key. The district panel uses
    Monday-start ISO weeks while the divisional roll-up uses weeks ending Saturday,
    so merging on a date column silently returns zero rows.
    """
    rows = []
    panel = panel.copy()
    panel["k"] = _isokey(panel["week_start"])

    # --- 1. cases vs the independent daily divisional bulletin
    bp = _find("divisional_daily_2022_2025.csv", "Dataset (1).csv", "dataset (1).csv")
    if bp:
        b = pd.read_csv(bp)
        b.columns = [c.strip().lower() for c in b.columns]
        b["date"] = pd.to_datetime(b["date"], format="mixed", errors="coerce")
        b = b.dropna(subset=["date"]); b["k"] = _isokey(b["date"])
        bw = b.groupby(["division", "k"])["patients"].sum().rename("bulletin").reset_index()
        pw = panel.groupby(["block", "k"])["cases"].sum().rename("panel").reset_index()                   .rename(columns={"block": "division"})
        m = pw.merge(bw, on=["division", "k"])
        m = m[m["k"].map(lambda t: 2022 <= t[0] <= 2025)]
        if len(m):
            rows += [
                {"check": "cases vs independent bulletin", "metric": "Pearson r",
                 "value": round(float(m.panel.corr(m.bulletin)), 4),
                 "n": len(m), "verdict": "PASS" if m.panel.corr(m.bulletin) > 0.99 else "FLAG"},
                {"check": "cases vs independent bulletin", "metric": "exact-match rate",
                 "value": round(float((m.panel == m.bulletin).mean()), 4), "n": len(m),
                 "verdict": "INFO"},
                {"check": "cases vs independent bulletin", "metric": "total difference %",
                 "value": round(float(100 * (m.panel.sum() - m.bulletin.sum()) / m.bulletin.sum()), 3),
                 "n": len(m), "verdict": "PASS" if abs(m.panel.sum() - m.bulletin.sum()) /
                 m.bulletin.sum() < 0.02 else "FLAG"}]
    else:
        rows.append({"check": "cases vs independent bulletin", "metric": "not run",
                     "value": np.nan, "n": 0, "verdict": "SKIPPED — divisional file not attached"})

    # --- 2. climate vs an independent NASA POWER harvest
    np_path = _find("nasa_power_divisions_daily.csv")
    if np_path:
        n = pd.read_csv(np_path)
        n["date"] = pd.to_datetime(n["date"], errors="coerce"); n["k"] = _isokey(n["date"])
        nw = (n.groupby(["division", "k"])
                .agg(t_nasa=("temp_mean", "mean"), h_nasa=("humidity", "mean"),
                     r_nasa=("rain", "sum")).reset_index())
        dc = (panel.groupby(["block", "k"])
                .agg(t_panel=("temp_mean", "mean"), h_panel=("humidity", "mean"),
                     r_panel=("rain", "mean")).reset_index().rename(columns={"block": "division"}))
        c = dc.merge(nw, on=["division", "k"])
        for a, b_, lab, thr in [("t_panel", "t_nasa", "temp_mean", 0.97),
                                ("h_panel", "h_nasa", "humidity", 0.95),
                                ("r_panel", "r_nasa", "rain", 0.95)]:
            r_ = float(c[a].corr(c[b_])) if len(c) else np.nan
            rows.append({"check": f"climate vs NASA POWER: {lab}", "metric": "Pearson r",
                         "value": round(r_, 4), "n": len(c),
                         "verdict": "PASS" if r_ > thr else
                         "FLAG — different product (panel rain is CHIRPS, not POWER)"})
            rows.append({"check": f"climate vs NASA POWER: {lab}", "metric": "mean bias (panel - NASA)",
                         "value": round(float((c[a] - c[b_]).mean()), 3), "n": len(c), "verdict": "INFO"})
    else:
        rows.append({"check": "climate vs NASA POWER", "metric": "not run", "value": np.nan,
                     "n": 0, "verdict": "SKIPPED — nasa_power_divisions_daily.csv not attached"})
    return pd.DataFrame(rows)


xc = save_table("table1c_external_cross_checks", external_cross_checks(PANEL_D),
                "Panel validated against sources it was not derived from")
print(xc.to_string(index=False))

zero_share = (PANEL_D.cases == 0).mean()
log.info("district zero-case share: %.1f%%  |  cumulative cases: %s",
         100 * zero_share, f"{int(PANEL_D.cases.sum()):,}")


In [ ]:
# =============================================================================
# CELL 3 — Feature engineering. Season-blocked so lags never bridge a data gap.
# =============================================================================
CLIM_BASE = ["rain", "temp_mean", "temp_min", "temp_max", "humidity"]


def engineer(panel):
    """Add lags, growth deltas, rolling stats, harmonics and targets.

    Lags are computed within (unit, block) where `block` marks a contiguous
    stretch of weeks. The district panel jumps 2019 -> 2022, so a plain shift
    would make the first week of 2022 look back at the last week of 2019.
    """
    df = panel.copy()
    df["gap_block"] = np.where(df["year"] <= 2019, "pre", "post")
    out = []
    for _, g in df.groupby(["unit", "gap_block"], sort=False):
        g = g.sort_values("week_start").copy()
        g["cases_lag0"] = g["cases"]
        g["log_cases_lag0"] = np.log1p(g["cases"])
        for L in [1, 2, 3, 4, 8]:
            g[f"cases_lag{L}"] = g["cases"].shift(L)
            g[f"log_cases_lag{L}"] = np.log1p(g[f"cases_lag{L}"])
        g["growth_delta_1w"] = g["log_cases_lag0"] - g["log_cases_lag1"]
        g["growth_delta_2w"] = g["log_cases_lag0"] - g["log_cases_lag2"]
        g["growth_delta_4w"] = g["log_cases_lag0"] - g["log_cases_lag4"]
        g["rolling_mean_log_4w"] = g["log_cases_lag0"].rolling(4, min_periods=2).mean()
        g["rolling_std_log_4w"]  = g["log_cases_lag0"].rolling(4, min_periods=2).std().fillna(0)
        g["rolling_mean_log_8w"] = g["log_cases_lag0"].rolling(8, min_periods=4).mean()
        g["dev_from_mean_4w"]    = g["log_cases_lag0"] - g["rolling_mean_log_4w"]
        # Climate lags recomputed here rather than taken from the deposit, so the
        # definition is byte-identical at both resolutions.
        for c in CLIM_BASE:
            if c in g.columns:
                for L in (2, 3, 4):
                    g[f"{c}_ownlag{L}"] = g[c].shift(L)
        for h in HORIZONS:
            g[f"target_lead_{h}w"]   = g["cases"].shift(-h)
            g[f"target_growth_{h}w"] = np.log((g["cases"].shift(-h) + 1) / (g["cases"] + 1))
        out.append(g)
    df = pd.concat(out).reset_index(drop=True)
    df["sin_epi_week"] = np.sin(2 * np.pi * df["epi_week"] / 52.1775)
    df["cos_epi_week"] = np.cos(2 * np.pi * df["epi_week"] / 52.1775)

    # Alarm label — threshold from TRAINING years only, so the test period never
    # participates in defining its own labels.
    q = (df[df.year <= ALARM_TRAIN_MAX].groupby("unit")["cases"].quantile(ALARM_QUANTILE))
    thr = np.maximum(df["unit"].map(q).fillna(ALARM_MIN_CASES), ALARM_MIN_CASES)
    df["alarm_threshold"] = thr
    for h in HORIZONS:
        lead = df[f"target_lead_{h}w"]
        df[f"target_alarm_{h}w"] = (lead >= thr).astype(float).where(lead.notna())
    return df


# ---- Feature groups. An ALLOW-LIST: nothing enters the model unless named here.
G_AR = ["cases_lag0", "log_cases_lag0",
        "cases_lag1", "cases_lag2", "cases_lag3", "cases_lag4", "cases_lag8",
        "log_cases_lag1", "log_cases_lag2", "log_cases_lag3", "log_cases_lag4", "log_cases_lag8",
        "growth_delta_1w", "growth_delta_2w", "growth_delta_4w",
        "rolling_mean_log_4w", "rolling_std_log_4w", "rolling_mean_log_8w", "dev_from_mean_4w"]
G_SEASON   = ["sin_epi_week", "cos_epi_week", "epi_week"]
# Shared climate: base variables plus lags recomputed identically on both panels.
# Source is NASA POWER (+ CHIRPS precipitation in the district deposit) on both sides.
G_CLIMATE  = (CLIM_BASE
              + [f"{c}_ownlag{L}" for c in CLIM_BASE for L in (2, 3, 4)])
# District-only extras shipped by the deposit (its own lag definitions, plus CHIRPS
# precipitation power). Excluded from the shared set so the ablation stays like-for-like.
G_CLIMATE_EXTRA = ["precip_power", "rain_lag2", "rain_lag3", "rain_lag4",
                   "temp_mean_lag2", "temp_mean_lag3", "temp_mean_lag4",
                   "humidity_lag2", "humidity_lag3", "humidity_lag4"]
G_SAT      = ["ndvi", "ndwi", "lst", "viirs", "ndvi_filled", "viirs_change"]
G_STATIC   = ["pop_density", "river_density", "road_density", "population"]
G_TRENDS   = ["gtrends_dengue", "gtrends_dengue_nat", "gtrends_dengue_bn_nat", "gtrends_fever_bn_nat"]
G_MOBILITY = ["log_nbr_cases", "log_nbr_cases_lag1", "log_nbr_cases_lag2",
              "log_upstream_pressure", "log_upstream_pressure_lag1", "log_upstream_pressure_lag2",
              "festival_window", "festival_influx"]
G_SURV     = ["new_deaths", "currently_admitted", "incidence"]

# Never features. `ignition*` and `eligible*` are the deposit's own forward-looking
# labels; the cumulative counters are monotone season identifiers that will not transfer.
FORBIDDEN = {"ignition", "ignition_loose", "eligible", "eligible_loose", "naive",
             "cumulative_cases", "cumulative_deaths", "alarm_threshold",
             "cases", "unit", "block", "year", "week_start", "gap_block", "district_id"}

SHARED_GROUPS = [G_AR, G_SEASON, G_CLIMATE]                       # available at BOTH resolutions
FULL_GROUPS   = SHARED_GROUPS + [G_CLIMATE_EXTRA, G_SAT, G_STATIC, G_TRENDS, G_MOBILITY, G_SURV]

def select_features(df, groups):
    allow, seen = [], set()
    for g in groups:
        for c in g:
            if c in df.columns and c not in FORBIDDEN and c not in seen:
                if pd.api.types.is_numeric_dtype(df[c]):
                    allow.append(c); seen.add(c)
    return allow


def prepare(panel, groups, min_year=2022):
    df = engineer(panel)
    F = select_features(df, groups)
    M = df[(df.year >= min_year) & df[f"target_growth_1w"].notna() & df["cases_lag8"].notna()].copy()
    M[F] = M[F].fillna(0.0)
    return M.reset_index(drop=True), F


MD, FD_FULL = prepare(PANEL_D, FULL_GROUPS)          # district, all covariates
_,  FD_SHR  = prepare(PANEL_D, SHARED_GROUPS)        # district, shared covariates only
MV, FV_SHR  = (prepare(PANEL_V, SHARED_GROUPS) if PANEL_V is not None else (None, []))

log.info("district model frame: %s rows | %d full features | %d shared features",
         f"{len(MD):,}", len(FD_FULL), len(FD_SHR))
if MV is not None:
    log.info("division model frame: %s rows | %d shared features", f"{len(MV):,}", len(FV_SHR))


In [ ]:
# =============================================================================
# CELL 4 — Integrity gate. Every defect from the 4 Sep review trips one of these.
# =============================================================================
def gate_data(M, F, recon_df, tag):
    assert not (FORBIDDEN & set(F)), f"[{tag}] forbidden column in feature set"
    assert not any(c.startswith("target_") for c in F), f"[{tag}] target leaked into features"
    bad = recon_df[(recon_df.panel == tag) & (recon_df.status == "DISCREPANCY")]
    assert bad.empty, f"[{tag}] reconciliation failed:\n{bad}"
    # no lag may reach across the 2020-21 gap
    early = M[(M.year == 2022) & (M.epi_week <= 8)]
    if len(early):
        assert early["cases_lag8"].notna().sum() == 0 or "pre" not in set(M.gap_block), \
            f"[{tag}] lag features bridge the 2020-21 gap"
    assert M.groupby(["unit", "week_start"]).size().max() == 1, f"[{tag}] duplicate unit-week rows"
    log.info("[gate] %-10s OK — %d features, %s rows", tag, len(F), f"{len(M):,}")

gate_data(MD, FD_FULL, recon, "district")
if MV is not None:
    gate_data(MV, FV_SHR, recon, "divisional")
    missing = set(FD_SHR) ^ set(FV_SHR)
    assert not missing, ("resolution ablation is not like-for-like; feature sets differ by "
                         f"{sorted(missing)}")
    log.info("[gate] shared feature set identical at both resolutions (%d features)", len(FD_SHR))


In [ ]:
# =============================================================================
# CELL 4b — Empirical model selection on an INNER split (never sees a test year)
# =============================================================================
# Selection train = 2019 tail + 2022, selection validation = 2023.
# 2024 and 2025 are untouched here, so nothing downstream is tuned on its own test set.
from sklearn.model_selection import ParameterGrid

MSEL, FSEL = prepare(PANEL_D, FULL_GROUPS, min_year=2019)
SEL_TRAIN, SEL_VALID = 2022, 2023
_tr = MSEL[(MSEL.year <= SEL_TRAIN) & MSEL["target_lead_2w"].notna()]
_va = MSEL[(MSEL.year == SEL_VALID) & MSEL["target_lead_2w"].notna()]
log.info("selection split: train %s rows (<=%d), validate %s rows (%d)",
         f"{len(_tr):,}", SEL_TRAIN, f"{len(_va):,}", SEL_VALID)

def _score(params, target, extra=None, h=2):
    """MAE on the inner validation year. h=2 is the operational horizon; using one
    horizon for selection avoids tuning the selection itself to the reported grid."""
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    col = tl if target == "level" else tg
    m = lgb.LGBMRegressor(**{**params, "random_state": SEED, "verbose": -1, "n_jobs": -1},
                          **(extra or {})).fit(_tr[FSEL], _tr[col])
    pred = m.predict(_va[FSEL])
    if target == "growth":
        pred = (_va["cases_lag0"].values + 1.0) * np.exp(pred) - 1.0
    return mean_absolute_error(_va[tl], np.clip(pred, 0, None))

GRID = list(ParameterGrid({"num_leaves": [15, 31, 63], "min_child_samples": [10, 20, 40],
                           "learning_rate": [0.05], "n_estimators": [300],
                           "max_depth": [5], "subsample": [0.8], "colsample_bytree": [0.8]}))
sel_rows = []

# (a) tree hyperparameters, chosen once on the growth target and reused for both,
#     so the two paradigms cannot differ merely through their tuning budget.
best = min(GRID, key=lambda g: _score(g, "growth"))
for g in GRID:
    sel_rows.append({"decision": "tree hyperparameters", "candidate": str(
        {k: g[k] for k in ("num_leaves", "min_child_samples")}),
        "inner_valid_MAE": round(_score(g, "growth"), 4),
        "selected": g == best})
LGB_REG = {**best, "random_state": SEED, "verbose": -1, "n_jobs": -1}
log.info("selected hyperparameters: %s", {k: best[k] for k in ("num_leaves", "min_child_samples")})

# (b) Tweedie variance power for the level target
tw_scores = {q: _score(best, "level", dict(objective="tweedie", tweedie_variance_power=q))
             for q in [1.1, 1.3, 1.5, 1.7, 1.9]}
TWEEDIE_P = min(tw_scores, key=tw_scores.get)
sel_rows += [{"decision": "tweedie_variance_power", "candidate": str(q),
              "inner_valid_MAE": round(v, 4), "selected": q == TWEEDIE_P}
             for q, v in tw_scores.items()]
log.info("selected tweedie_variance_power: %.1f", TWEEDIE_P)

# (c) objective for the anchored-growth target
gr_scores = {"L2": _score(best, "growth", dict(objective="regression")),
             "L1": _score(best, "growth", dict(objective="regression_l1"))}
GROWTH_OBJ_NAME = min(gr_scores, key=gr_scores.get)
sel_rows += [{"decision": "growth objective", "candidate": k, "inner_valid_MAE": round(v, 4),
              "selected": k == GROWTH_OBJ_NAME} for k, v in gr_scores.items()]
log.info("selected growth objective: %s", GROWTH_OBJ_NAME)

# (d) conformal calibration window, scored by |coverage - nominal| on the inner split
def _cal_score(cw, h=2):
    tg, tl = f"target_growth_{h}w", f"target_lead_{h}w"
    tra = _tr.sort_values("week_start")
    cut = tra["week_start"].drop_duplicates().nlargest(cw).min()
    tr2, cal = tra[tra.week_start < cut], tra[tra.week_start >= cut]
    if len(tr2) < 300 or len(cal) < 100:
        return np.inf
    q = {a: lgb.LGBMRegressor(**LGB_REG, objective="quantile", alpha=a).fit(tr2[FSEL], tr2[tg])
         for a in (0.05, 0.95)}
    sc = np.maximum(q[0.05].predict(cal[FSEL]) - cal[tg].values,
                    cal[tg].values - q[0.95].predict(cal[FSEL]))
    n = len(sc); qh = np.quantile(sc, min(1.0, np.ceil((n + 1) * NOMINAL) / n))
    anc = _va["cases_lag0"].values + 1.0
    lo = anc * np.exp(q[0.05].predict(_va[FSEL]) - qh) - 1
    hi = anc * np.exp(q[0.95].predict(_va[FSEL]) + qh) - 1
    y = _va[tl].values
    return abs(((y >= np.clip(lo, 0, None)) & (y <= hi)).mean() - NOMINAL)

# Selected PER HORIZON: the score is |coverage - nominal|, and a 1-week-ahead
# residual distribution is not the same object as a 4-week-ahead one.
CAL_WEEKS_BY_H = {}
for _h in [1, 2, 4]:
    _sc = {cw: _cal_score(cw, h=_h) for cw in [8, 13, 26, 39]}
    CAL_WEEKS_BY_H[_h] = min(_sc, key=_sc.get)
    sel_rows += [{"decision": f"conformal calibration weeks (h={_h})", "candidate": str(cw),
                  "inner_valid_MAE": round(v, 4), "selected": cw == CAL_WEEKS_BY_H[_h]}
                 for cw, v in _sc.items()]
CAL_WEEKS = CAL_WEEKS_BY_H[2]
log.info("selected conformal calibration window per horizon: %s", CAL_WEEKS_BY_H)

save_table("table1d_model_selection", pd.DataFrame(sel_rows),
           f"Every tuned constant, chosen on train<={SEL_TRAIN} / validate {SEL_VALID}; "
           f"metric is inner-validation MAE except the calibration window (|coverage-nominal|)")
del MSEL, FSEL, _tr, _va


In [ ]:
# =============================================================================
# CELL 5 — Metrics, rolling-origin harness, and the model zoo
# =============================================================================
def metrics(y, p):
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 0, None)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    return dict(MAE=mean_absolute_error(y, p),
                RMSE=float(np.sqrt(np.mean((y - p) ** 2))),
                wMAPE=float(100 * np.abs(y - p).sum() / y.sum()) if y.sum() > 0 else np.nan)

# Each target gets a sweep of plausible objectives, so neither side of the
# comparison can be accused of running on a misspecified likelihood.
LEVEL_OBJECTIVES = {
    "L2":      dict(objective="regression"),
    "Poisson": dict(objective="poisson"),
    "Tweedie": dict(objective="tweedie", tweedie_variance_power=TWEEDIE_P),
}
GROWTH_OBJECTIVES = {
    "L2": dict(objective="regression"),
    "L1": dict(objective="regression_l1"),   # median of the log-ratio
}

def rolling_origin(M, F, h, test_years=TEST_YEARS):
    """Expanding-window rolling origin. Returns one long frame of aligned predictions."""
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    frames = []
    for ty in test_years:
        tr = M[(M.year < ty) & M[tl].notna()]
        te = M[(M.year == ty) & M[tl].notna()]
        if len(te) < 20 or len(tr) < 100:
            continue
        out = te[["unit", "block", "year", "epi_week", "week_start", "cases_lag0", tl]].copy()
        out = out.rename(columns={tl: "y"})
        out["fold"] = ty
        out["persistence"] = te["cases_lag0"].values

        for name, obj in LEVEL_OBJECTIVES.items():
            m = lgb.LGBMRegressor(**LGB_REG, **obj).fit(tr[F], tr[tl])
            out[f"level_{name}"] = np.clip(m.predict(te[F]), 0, None)

        for name, obj in GROWTH_OBJECTIVES.items():
            m = lgb.LGBMRegressor(**LGB_REG, **obj).fit(tr[F], tr[tg])
            out[f"anchored_{name}"] = np.clip(
                (te["cases_lag0"].values + 1.0) * np.exp(m.predict(te[F])) - 1.0, 0, None)
        out["anchored"] = out[f"anchored_{GROWTH_OBJ_NAME}"]   # objective chosen in Cell 4b

        # Seasonal naive: same ISO week one year earlier, matched on (unit, epi_week).
        prev = (M[(M.year == ty - 1)].groupby(["unit", "epi_week"])["cases"].mean()
                  .rename("snaive").reset_index())
        out = out.merge(prev, on=["unit", "epi_week"], how="left")
        out["snaive"] = out["snaive"].fillna(out["persistence"])

        # Log-AR(4) ridge, a linear floor.
        ar = [c for c in ["log_cases_lag0", "log_cases_lag1", "log_cases_lag2", "log_cases_lag3"] if c in F]
        if ar:
            r = Ridge(alpha=10.0).fit(tr[ar], np.log1p(tr[tl]))
            out["ridge_ar4"] = np.clip(np.expm1(r.predict(te[ar])), 0, None)
        else:
            out["ridge_ar4"] = out["persistence"]

        frames.append(out)
    return pd.concat(frames, ignore_index=True)


PRED = {h: rolling_origin(MD, FD_FULL, h) for h in HORIZONS}
log.info("rolling-origin predictions built for h = %s", HORIZONS)

MODELS = ["persistence", "snaive", "ridge_ar4",
          "level_L2", "level_Poisson", "level_Tweedie",
          "anchored_L2", "anchored_L1"]
LABELS = {"persistence": "Lag-0 persistence", "snaive": "Seasonal naive",
          "ridge_ar4": "Log-AR(4) ridge",
          "level_L2": "LightGBM level (L2)",
          "level_Poisson": "LightGBM level (Poisson)",
          "level_Tweedie": "LightGBM level (Tweedie)",
          "anchored_L2": "LightGBM anchored growth (L2)",
          "anchored_L1": "LightGBM anchored growth (L1)",
          "anchored": "LightGBM anchored growth (L1)"}

rows = []
for h in HORIZONS:
    P = PRED[h]; base = mean_absolute_error(P.y, P.persistence)
    for m in MODELS:
        mm = metrics(P.y, P[m])
        rows.append({"horizon_weeks": h, "model": LABELS[m], "key": m,
                     "MAE": round(mm["MAE"], 3), "RMSE": round(mm["RMSE"], 3),
                     "wMAPE_pct": round(mm["wMAPE"], 2),
                     "skill_vs_persistence_pct": round(100 * (1 - mm["MAE"] / base), 2)})
shootout = save_table("table2_forecast_shootout", pd.DataFrame(rows),
                      "Multi-horizon prospective forecast comparison, district resolution")
print(shootout.pivot_table(index="horizon_weeks", columns="model",
                           values="skill_vs_persistence_pct").round(1).to_string())


In [ ]:
# =============================================================================
# CELL 6 — Block bootstrap + Diebold–Mariano on the headline deltas
# =============================================================================
def block_bootstrap_delta(P, a, b, n_boot=2000, seed=SEED):
    """95% interval on MAE(a) - MAE(b), resampling whole (unit, fold) blocks."""
    rng = np.random.default_rng(seed)
    blocks = list(P.groupby(["unit", "fold"]).indices.values())
    y, pa, pb = P.y.values, P[a].values, P[b].values
    draws = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))])
        draws[i] = np.abs(y[idx] - pa[idx]).mean() - np.abs(y[idx] - pb[idx]).mean()
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


def panel_dm(P, a, b):
    """Panel Diebold-Mariano.

    Each unit contributes ONE number - its mean absolute-error differential - and the
    test is a one-sample t-test across units. Collapsing to the unit level absorbs the
    serial correlation that a multi-step forecast necessarily induces (an h-step error
    sequence is MA(h-1)), so no HAC correction is needed and the cross-sectional
    dependence is handled by treating units as the sampling unit.
    """
    d = (P.assign(_d=np.abs(P.y - P[a]) - np.abs(P.y - P[b]))
           .groupby("unit")["_d"].mean())
    if len(d) < 3 or d.std(ddof=1) == 0:
        return np.nan, np.nan, len(d)
    t, pv = stats.ttest_1samp(d.values, 0.0)
    return float(t), float(pv), int(len(d))


def bh_fdr(pvals, q=ALPHA):
    """Benjamini-Hochberg. Returns the boolean reject vector and adjusted p-values."""
    p = np.asarray(pvals, float)
    ok = np.isfinite(p)
    adj = np.full_like(p, np.nan)
    if ok.sum() == 0:
        return np.zeros_like(p, bool), adj
    idx = np.where(ok)[0][np.argsort(p[ok])]
    n = len(idx)
    prev = 1.0
    for rank in range(n - 1, -1, -1):
        prev = min(prev, p[idx[rank]] * n / (rank + 1))
        adj[idx[rank]] = prev
    return (adj <= q) & ok, adj


rows = []
for h in HORIZONS:
    P = PRED[h]
    for a, b in [("anchored_L1", "persistence"), ("level_Tweedie", "persistence"),
                 ("anchored_L1", "level_Tweedie"), ("anchored_L1", "anchored_L2")]:
        lo, hi = block_bootstrap_delta(P, a, b)
        t, pv, nu = panel_dm(P, a, b)
        rows.append({"horizon_weeks": h, "comparison": f"{LABELS[a]} vs {LABELS[b]}",
                     "delta_MAE": round(mean_absolute_error(P.y, P[a]) - mean_absolute_error(P.y, P[b]), 3),
                     "boot_lo": round(lo, 3), "boot_hi": round(hi, 3),
                     "boot_excludes_0": bool(lo * hi > 0),
                     "DM_t": round(t, 3), "DM_p_raw": round(pv, 4), "n_units": nu})
unc = pd.DataFrame(rows)
# One family of tests, so the p-values are corrected together.
rej, adj = bh_fdr(unc["DM_p_raw"].values, q=ALPHA)
unc["DM_p_BH"] = np.round(adj, 4)
unc["significant"] = np.where(rej & unc["boot_excludes_0"], f"yes (BH q<{ALPHA})",
                       np.where(rej, "DM only - bootstrap spans 0",
                       np.where(unc["boot_excludes_0"], "bootstrap only - DM n.s.", "no")))
uncert = save_table("table3_uncertainty_on_deltas", unc,
                    f"Block-bootstrap intervals and panel Diebold-Mariano tests, "
                    f"Benjamini-Hochberg corrected at q={ALPHA}")
print(uncert[["horizon_weeks", "comparison", "delta_MAE", "boot_lo", "boot_hi",
              "DM_p_raw", "DM_p_BH", "significant"]].to_string(index=False))


In [ ]:
# =============================================================================
# CELL 7 — Prediction intervals: empirical baseline, split conformal, adaptive
# =============================================================================
def _cal_split(M, tg, ty, cal_weeks):
    tra = M[(M.year < ty) & M[tg].notna()].sort_values("week_start")
    cut = tra["week_start"].drop_duplicates().nlargest(cal_weeks).min()
    return tra[tra.week_start < cut], tra[tra.week_start >= cut]


def wis(y, lo, hi, med, alpha=1 - NOMINAL):
    """Weighted interval score for a single central interval plus a median."""
    y, lo, hi, med = map(lambda v: np.asarray(v, float), (y, lo, hi, med))
    interval = (hi - lo) + (2 / alpha) * (lo - y) * (y < lo) + (2 / alpha) * (y - hi) * (y > hi)
    return float(np.mean((0.5 * np.abs(y - med) + (alpha / 2) * interval) / 1.5))


def interval_run(M, F, h, method, test_years=TEST_YEARS, gamma=0.02):
    """method in {'empirical','raw','split','adaptive'}. Returns per-row interval frame."""
    tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
    frames = []
    for ty in test_years:
        te = M[(M.year == ty) & M[tl].notna()].sort_values("week_start")
        if len(te) < 20:
            continue
        anchor = te["cases_lag0"].values + 1.0
        y = te[tl].values

        if method == "empirical":
            # No model: per-unit empirical quantiles of historical h-step log growth.
            tr = M[(M.year < ty) & M[tg].notna()]
            ql = tr.groupby("unit")[tg].quantile(0.05); qh = tr.groupby("unit")[tg].quantile(0.95)
            qm = tr.groupby("unit")[tg].median()
            gl = te["unit"].map(ql).fillna(tr[tg].quantile(0.05)).values
            gh = te["unit"].map(qh).fillna(tr[tg].quantile(0.95)).values
            gm = te["unit"].map(qm).fillna(tr[tg].median()).values
            lo, hi, med = anchor * np.exp(gl) - 1, anchor * np.exp(gh) - 1, anchor * np.exp(gm) - 1
        else:
            tr, cal = _cal_split(M, tg, ty, CAL_WEEKS_BY_H.get(h, CAL_WEEKS))
            q = {a: lgb.LGBMRegressor(**LGB_REG, objective="quantile", alpha=a).fit(tr[F], tr[tg])
                 for a in (0.05, 0.50, 0.95)}   # quantile loss == pinball; L1 at a=0.50
            s = np.maximum(q[0.05].predict(cal[F]) - cal[tg].values,
                           cal[tg].values - q[0.95].predict(cal[F]))
            g_lo, g_hi, g_md = (q[0.05].predict(te[F]), q[0.95].predict(te[F]), q[0.50].predict(te[F]))

            if method == "raw":
                qh_vec = np.zeros(len(te))
            elif method == "split":
                n = len(s)
                qh_vec = np.full(len(te), np.quantile(s, min(1.0, np.ceil((n + 1) * NOMINAL) / n)))
            else:  # adaptive conformal (Gibbs & Candes): update the level week by week
                qh_vec = np.empty(len(te)); alpha_t = 1 - NOMINAL
                weeks = te["week_start"].values
                for wk in pd.unique(weeks):
                    sel = weeks == wk
                    lvl = float(np.clip(1 - alpha_t, 0.01, 0.999))
                    qh_wk = np.quantile(s, lvl)
                    qh_vec[sel] = qh_wk
                    cov_wk = ((y[sel] >= anchor[sel] * np.exp(g_lo[sel] - qh_wk) - 1) &
                              (y[sel] <= anchor[sel] * np.exp(g_hi[sel] + qh_wk) - 1)).mean()
                    alpha_t = alpha_t + gamma * ((1 - NOMINAL) - (1 - cov_wk))
            lo = anchor * np.exp(g_lo - qh_vec) - 1
            hi = anchor * np.exp(g_hi + qh_vec) - 1
            med = anchor * np.exp(g_md) - 1

        f = te[["unit", "block", "year", "epi_week", "week_start"]].copy()
        f["y"] = y; f["lo"] = np.clip(lo, 0, None); f["hi"] = hi
        f["med"] = np.clip(med, 0, None); f["fold"] = ty
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


INT_METHODS = ["empirical", "raw", "split", "adaptive"]
INT_LABEL = {"empirical": "Empirical growth quantiles (no model)",
             "raw": "Quantile LightGBM (uncalibrated)",
             "split": "Split-conformal CQR",
             "adaptive": "Adaptive conformal (ACI)"}

INTERVALS = {}
rows = []
for h in [1, 2, 4]:
    for meth in INT_METHODS:
        I = interval_run(MD, FD_FULL, h, meth)
        INTERVALS[(h, meth)] = I
        cov = float(((I.y >= I.lo) & (I.y <= I.hi)).mean())
        rows.append({"horizon_weeks": h, "method": INT_LABEL[meth],
                     "key": meth, "nominal": NOMINAL,
                     "empirical_coverage": round(cov, 4),
                     "median_width_cases": round(float(np.median(I.hi - I.lo)), 2),
                     "mean_width_cases": round(float(np.mean(I.hi - I.lo)), 2),
                     "WIS": round(wis(I.y, I.lo, I.hi, I.med), 3)})
def coverage_test(I, nominal=NOMINAL):
    """Is empirical coverage different from nominal? Unit-level coverage rates, t-test.

    Each district contributes one coverage proportion, so within-district dependence
    does not inflate the test the way a row-level binomial test would.
    """
    cov = (I.assign(_c=((I.y >= I.lo) & (I.y <= I.hi)).astype(float))
             .groupby("unit")["_c"].mean())
    t, pv = stats.ttest_1samp(cov.values, nominal)
    return float(cov.mean()), float(t), float(pv), int(len(cov))

for r in rows:
    I = INTERVALS[(r["horizon_weeks"], r["key"])]
    _, t, pv, nu = coverage_test(I)
    r["cov_t"] = round(t, 3); r["cov_p_raw"] = round(pv, 4); r["n_units"] = nu
_c = pd.DataFrame(rows)
_rej, _adj = bh_fdr(_c["cov_p_raw"].values, q=ALPHA)
_c["cov_p_BH"] = np.round(_adj, 4)
# For a calibration claim the DESIRABLE outcome is failing to reject: coverage
# indistinguishable from nominal. Stating it this way round avoids reading a
# non-significant result as evidence of nothing.
_c["verdict"] = np.where(_rej, f"differs from {NOMINAL:.2f} (BH q<{ALPHA})",
                         f"indistinguishable from {NOMINAL:.2f}")
rows = _c.to_dict("records")
calib = save_table("table4_interval_calibration", _c,
                   "Coverage, width, WIS, and a test of coverage against nominal")
print(calib.pivot_table(index="horizon_weeks", columns="key",
                        values="empirical_coverage").round(3).to_string())


In [ ]:
# =============================================================================
# CELL 8 — Conditional coverage by district burden. Marginal coverage hides this.
# =============================================================================
burden = MD[MD.year <= ALARM_TRAIN_MAX].groupby("unit")["cases"].mean()
tert = pd.qcut(burden, 3, labels=["low burden", "mid burden", "high burden"])

rows = []
for h in [1, 2, 4]:
    for meth in ["raw", "split", "adaptive"]:
        I = INTERVALS[(h, meth)].copy()
        I["tertile"] = I["unit"].map(tert)
        for t, g in I.groupby("tertile", observed=True):
            rows.append({"horizon_weeks": h, "method": INT_LABEL[meth], "burden_tertile": str(t),
                         "n_unit_weeks": len(g),
                         "coverage": round(float(((g.y >= g.lo) & (g.y <= g.hi)).mean()), 4),
                         "median_width_cases": round(float(np.median(g.hi - g.lo)), 2)})
cond = save_table("table5_conditional_coverage", pd.DataFrame(rows),
                  "Coverage conditional on district burden tertile")
print(cond[cond.horizon_weeks == 2].to_string(index=False))


In [ ]:
# =============================================================================
# CELL 9 — 2x2 space-time validation matrix, spatial folds rotated over all blocks
# =============================================================================
H_ALARM = 2
A = MD[MD[f"target_alarm_{H_ALARM}w"].notna()].copy()
YA = f"target_alarm_{H_ALARM}w"
log.info("alarm task: n=%s, base rate %.3f", f"{len(A):,}", A[YA].mean())

def _fit(tr, te, F=FD_FULL):
    m = lgb.LGBMClassifier(**LGB_CLF).fit(tr[F], tr[YA].astype(int))
    return m.predict_proba(te[F])[:, 1], te[YA].astype(int).values

def _pool(pairs):
    ps, ys = zip(*pairs)
    p, y = np.concatenate(ps), np.concatenate(ys)
    return float(roc_auc_score(y, p)), float(average_precision_score(y, p))

BLOCKS = sorted(A["block"].unique())
tr_c1, te_c1 = train_test_split(A, test_size=0.20, random_state=SEED, stratify=A[YA])
c1 = _pool([_fit(tr_c1, te_c1)])
c2 = _pool([_fit(A[A.year < ty], A[A.year == ty]) for ty in TEST_YEARS])
c3 = _pool([_fit(A[A.block != b], A[A.block == b]) for b in BLOCKS])
c4 = _pool([_fit(A[(A.year < ty) & (A.block != b)], A[(A.year == ty) & (A.block == b)])
            for ty in TEST_YEARS for b in BLOCKS])

matrix = save_table("table6_optimism_gap", pd.DataFrame([
    {"condition": "C1 random 80/20", "holds_out": "nothing meaningful", "ROC_AUC": round(c1[0], 4), "PR_AUC": round(c1[1], 4)},
    {"condition": "C2 rolling origin", "holds_out": "future seasons", "ROC_AUC": round(c2[0], 4), "PR_AUC": round(c2[1], 4)},
    {"condition": "C3 leave-one-division-out", "holds_out": "unseen geography", "ROC_AUC": round(c3[0], 4), "PR_AUC": round(c3[1], 4)},
    {"condition": "C4 space + time", "holds_out": "both", "ROC_AUC": round(c4[0], 4), "PR_AUC": round(c4[1], 4)},
    {"condition": "OPTIMISM GAP (C1 - C4)", "holds_out": "", "ROC_AUC": round(c1[0] - c4[0], 4), "PR_AUC": round(c1[1] - c4[1], 4)},
]), "2x2 space-time validation matrix")
print(matrix.to_string(index=False))

assert c4[0] <= min(c2[0], c3[0]) + 0.01, "validation matrix incoherent: C4 above C2 or C3"
log.info("[gate] validation matrix ordering OK")


In [ ]:
# =============================================================================
# CELL 10 — Operational alarms: fixed sensitivity, false-alarm rate, lead time
# =============================================================================
alarm_rows, lead_rows, div_rows = [], [], []
for h in HORIZONS:
    ycol = f"target_alarm_{h}w"
    sub = MD[MD[ycol].notna()]
    tr = sub[sub.year <= 2023]
    te = sub[sub.year.isin(TEST_YEARS)].copy()
    clf = lgb.LGBMClassifier(**LGB_CLF).fit(tr[FD_FULL], tr[ycol].astype(int))
    te["p"] = clf.predict_proba(te[FD_FULL])[:, 1]
    y, p = te[ycol].astype(int).values, te["p"].values
    prec, rec, thr = precision_recall_curve(y, p)

    for target in (0.80, 0.90):
        ok = np.where(rec[:-1] >= target)[0]
        if not len(ok):
            continue
        i = ok[-1]
        t = thr[i]
        pred = (p >= t).astype(int)
        fp = int(((pred == 1) & (y == 0)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
        alarm_rows.append({"horizon_weeks": h, "target_sensitivity": target,
                           "achieved_sensitivity": round(float(rec[i]), 3),
                           "precision": round(float(prec[i]), 3),
                           "false_alarm_rate": round(fp / (fp + tn), 4) if fp + tn else np.nan,
                           "threshold": round(float(t), 4)})
        if h == 2 and target == 0.80:
            # Lead time: weeks between the first alarm and that unit-season's peak.
            for (u, yr), g in te.groupby(["unit", "year"]):
                g = g.sort_values("week_start")
                # The event the system exists to anticipate: the season's first week
                # in which this district actually crosses its own outbreak threshold.
                crossed = g[g["cases_lag0"] >= g["alarm_threshold"]]
                if not len(crossed):
                    continue
                onset = crossed["week_start"].min()
                peak  = g.loc[g["cases_lag0"].idxmax(), "week_start"]
                # Genuine advance warning: the alarm must fire while the district is
                # still quiet, i.e. strictly before that first crossing. Without this
                # the metric only records when an already-endemic season began.
                fired = g[(g.p >= t) & (g.week_start < onset) &
                          (g["cases_lag0"] < g["alarm_threshold"])]
                if len(fired):
                    first = fired["week_start"].min()
                    lead_rows.append({
                        "unit": u, "season": int(yr),
                        "first_alarm": str(pd.Timestamp(first).date()),
                        "onset_week": str(pd.Timestamp(onset).date()),
                        "peak_week": str(pd.Timestamp(peak).date()),
                        "lead_weeks_to_onset": int((onset - first).days / 7),
                        "lead_weeks_to_peak": int((peak - first).days / 7)})
            for b, g in te.groupby("block"):
                if g[ycol].nunique() > 1:
                    div_rows.append({"division": b, "n_unit_weeks": len(g),
                                     "outbreak_rate": round(float(g[ycol].mean()), 3),
                                     "ROC_AUC": round(float(roc_auc_score(g[ycol], g.p)), 4),
                                     "PR_AUC": round(float(average_precision_score(g[ycol], g.p)), 4)})

save_table("table7a_alarm_performance", pd.DataFrame(alarm_rows),
           "Alarm performance at fixed public-health sensitivity")
LEAD = save_table("table7b_alarm_lead_time", pd.DataFrame(lead_rows),
                  "Weeks of warning before outbreak onset and before peak, h=2, sensitivity 0.80")
save_table("table7c_divisional_alarm", pd.DataFrame(div_rows),
           "Alarm discrimination by division, h=2")
if len(LEAD):
    log.info("lead time to outbreak onset: median %.0f wk (IQR %.0f-%.0f) | to peak: median %.0f wk | n=%d district-seasons",
             LEAD.lead_weeks_to_onset.median(), LEAD.lead_weeks_to_onset.quantile(.25),
             LEAD.lead_weeks_to_onset.quantile(.75), LEAD.lead_weeks_to_peak.median(), len(LEAD))


In [ ]:
# =============================================================================
# CELL 11 — Covariate-group ablation. Do satellite / trends / mobility earn a place?
# =============================================================================
ABLATION = {
    "case history + season":  G_AR + G_SEASON,
    "+ climate":              G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA,
    "+ satellite":            G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC,
    "+ search trends":        G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC + G_TRENDS,
    "+ mobility & spillover": G_AR + G_SEASON + G_CLIMATE + G_CLIMATE_EXTRA + G_SAT + G_STATIC + G_TRENDS + G_MOBILITY + G_SURV,
}
rows = []
for h in [1, 2, 4]:
    base = None
    for name, cols in ABLATION.items():
        F = select_features(MD, [cols])
        P = rolling_origin(MD, F, h)
        mae = mean_absolute_error(P.y, P.anchored)
        pers = mean_absolute_error(P.y, P.persistence)
        base = mae if base is None else base
        rows.append({"horizon_weeks": h, "feature_set": name, "n_features": len(F),
                     "MAE_anchored": round(mae, 3),
                     "skill_vs_persistence_pct": round(100 * (1 - mae / pers), 2),
                     "delta_MAE_vs_case_history": round(mae - base, 3)})
abl = save_table("table8_covariate_ablation", pd.DataFrame(rows),
                 "Marginal value of each covariate group, anchored-growth model")
print(abl[abl.horizon_weeks == 2].to_string(index=False))


In [ ]:
# =============================================================================
# CELL 12 — Resolution ablation. Same code path, shared features, both panels.
# =============================================================================
def resolution_summary(M, F, tag):
    out = []
    for h in HORIZONS:
        P = rolling_origin(M, F, h)
        if not len(P):
            continue
        pers = mean_absolute_error(P.y, P.persistence)
        for m in ["level_Tweedie", "level_L2", "anchored_L1", "anchored_L2"]:
            out.append({"resolution": tag, "n_units": M.unit.nunique(), "horizon_weeks": h,
                        "model": LABELS[m], "key": m,
                        "MAE": round(mean_absolute_error(P.y, P[m]), 3),
                        "skill_vs_persistence_pct": round(100 * (1 - mean_absolute_error(P.y, P[m]) / pers), 2)})
    return pd.DataFrame(out)

res = resolution_summary(MD, FD_SHR, "district (64 units)")
if MV is not None:
    res = pd.concat([res, resolution_summary(MV, FV_SHR, "divisional (8 units)")], ignore_index=True)
res = save_table("table9_resolution_ablation", res,
                 "Skill by spatial resolution, shared feature set only")

# Contribution 2 is a difference-in-differences claim: the anchored-minus-level skill
# gap is LARGER at coarse resolution. That needs a test, not two columns side by side.
def resolution_did(h=2, a="anchored_L1", b="level_Tweedie", n_boot=2000, seed=SEED,
                   fd=None, fv=None):
    if MV is None:
        return None
    Pd, Pv = rolling_origin(MD, fd or FD_SHR, h), rolling_origin(MV, fv or FV_SHR, h)
    if not len(Pd) or not len(Pv):
        return None
    rng = np.random.default_rng(seed)

    def gap(P, units):
        sub = P[P.unit.isin(units)]
        if not len(sub):
            return np.nan
        base = mean_absolute_error(sub.y, sub.persistence)
        if base == 0:
            return np.nan
        return (100 * (1 - mean_absolute_error(sub.y, sub[a]) / base)
                - 100 * (1 - mean_absolute_error(sub.y, sub[b]) / base))

    ud, uv = Pd.unit.unique(), Pv.unit.unique()
    obs = gap(Pv, uv) - gap(Pd, ud)
    draws = np.array([gap(Pv, rng.choice(uv, len(uv), replace=True))
                      - gap(Pd, rng.choice(ud, len(ud), replace=True)) for _ in range(n_boot)])
    draws = draws[np.isfinite(draws)]
    lo, hi = np.percentile(draws, [2.5, 97.5])
    pv = 2 * min((draws <= 0).mean(), (draws >= 0).mean())
    return {"horizon_weeks": h, "gap_divisional_pp": round(gap(Pv, uv), 2),
            "gap_district_pp": round(gap(Pd, ud), 2),
            "difference_in_differences_pp": round(float(obs), 2),
            "boot_lo": round(float(lo), 2), "boot_hi": round(float(hi), 2),
            "boot_p": round(float(pv), 4),
            "verdict": ("gap is larger at coarse resolution" if lo > 0 else
                        "gap is larger at fine resolution" if hi < 0 else
                        "no significant resolution effect")}

did = [d for d in (resolution_did(h) for h in HORIZONS) if d]
if did:
    D = pd.DataFrame(did)
    _rej, _adj = bh_fdr(D["boot_p"].values, q=ALPHA)
    D["boot_p_BH"] = np.round(_adj, 4)
    save_table("table9b_resolution_significance", D,
               "Difference-in-differences test of the resolution-dependence claim")
    print(D.to_string(index=False))
print(res.pivot_table(index=["horizon_weeks"], columns=["resolution", "key"],
                      values="skill_vs_persistence_pct").round(1).to_string())


In [ ]:
# =============================================================================
# CELL 12b — Sensitivity analyses for the two data-quality defects found in the audit
# =============================================================================
sens_rows = []

# --- A. Zeros that are probably MISSING REPORTS rather than true zeros -------
# table1b flags weeks with cases = 0 while patients are still admitted. If those
# rows drive the headline, the zero-inflation story is an artefact of missingness.
gap_key = set(map(tuple, PANEL_D.loc[
    (PANEL_D.cases == 0) & (PANEL_D.get("currently_admitted", 0) > 0),
    ["unit", "week_start"]].values))
log.info("report-gap weeks flagged: %d", len(gap_key))

def _is_gap(units, weeks):
    return np.array([(u, w) in gap_key for u, w in zip(units, weeks)])

for h in [1, 2]:
    P = PRED[h].copy()
    tgt_week = P["week_start"] + pd.to_timedelta(7 * h, unit="D")
    drop = _is_gap(P.unit.values, P.week_start.values) | _is_gap(P.unit.values, tgt_week.values)
    keep = P[~drop]
    for tag, sub in [("all rows", P), ("report-gap rows removed", keep)]:
        base = mean_absolute_error(sub.y, sub.persistence)
        sens_rows.append({
            "sensitivity": "A · missing-report zeros", "horizon_weeks": h, "variant": tag,
            "n_rows": len(sub), "rows_dropped_pct": round(100 * (1 - len(sub) / len(P)), 2),
            "anchored_skill_pct": round(100 * (1 - mean_absolute_error(sub.y, sub.anchored) / base), 2),
            "level_Tweedie_skill_pct": round(100 * (1 - mean_absolute_error(sub.y, sub.level_Tweedie) / base), 2)})
    I = INTERVALS[(h, "split")].copy()
    tw = I["week_start"] + pd.to_timedelta(7 * h, unit="D")
    dr = _is_gap(I.unit.values, I.week_start.values) | _is_gap(I.unit.values, tw.values)
    for tag, sub in [("all rows", I), ("report-gap rows removed", I[~dr])]:
        sens_rows.append({
            "sensitivity": "A · missing-report zeros (coverage)", "horizon_weeks": h, "variant": tag,
            "n_rows": len(sub), "rows_dropped_pct": round(100 * (1 - len(sub) / len(I)), 2),
            "anchored_skill_pct": round(float(((sub.y >= sub.lo) & (sub.y <= sub.hi)).mean()), 4),
            "level_Tweedie_skill_pct": np.nan})

# --- B. Rainfall is CHIRPS in the district panel, NASA POWER in the divisional one
# table1c flags this (r = 0.86). If the resolution result survives dropping rain
# entirely, the residual source mismatch is not what produces it.
RAIN_COLS = [c for c in FD_SHR if c.startswith("rain")]
fd_norain = [c for c in FD_SHR if c not in RAIN_COLS]
fv_norain = [c for c in FV_SHR if not c.startswith("rain")]
log.info("rain columns removed for sensitivity B: %s", RAIN_COLS)
for h in HORIZONS:
    for tag, kw in [("with rain", {}), ("rain removed", dict(fd=fd_norain, fv=fv_norain))]:
        d_ = resolution_did(h, n_boot=800, **kw)
        if d_:
            sens_rows.append({"sensitivity": "B · rain source mismatch", "horizon_weeks": h,
                              "variant": tag, "n_rows": np.nan, "rows_dropped_pct": np.nan,
                              "anchored_skill_pct": d_["difference_in_differences_pp"],
                              "level_Tweedie_skill_pct": np.nan,
                              "boot_lo": d_["boot_lo"], "boot_hi": d_["boot_hi"],
                              "boot_p": d_["boot_p"], "verdict": d_["verdict"]})

SENS = save_table("table11_data_quality_sensitivity", pd.DataFrame(sens_rows),
                  "Do the two audit defects change the conclusions? "
                  "A: missing-report zeros. B: CHIRPS-vs-POWER rainfall mismatch")
print(SENS.to_string(index=False))


In [ ]:
# =============================================================================
# CELL 13 — 2026 forward test. Frozen at end-2025, never refitted.
# =============================================================================
fwd_rows = []
if (MD.year == 2026).any():
    for h in [1, 2, 4]:
        tl, tg = f"target_lead_{h}w", f"target_growth_{h}w"
        tr_all = MD[(MD.year <= 2025) & MD[tg].notna()].sort_values("week_start")
        te = MD[(MD.year == 2026) & MD[tl].notna()]
        if len(te) < 20:
            continue
        cut = tr_all["week_start"].drop_duplicates().nlargest(CAL_WEEKS_BY_H.get(h, CAL_WEEKS)).min()
        tr, cal = tr_all[tr_all.week_start < cut], tr_all[tr_all.week_start >= cut]
        mg = lgb.LGBMRegressor(**LGB_REG, objective="regression").fit(tr[FD_FULL], tr[tg])
        anchor = te["cases_lag0"].values + 1.0
        pred = np.clip(anchor * np.exp(mg.predict(te[FD_FULL])) - 1, 0, None)
        q = {a: lgb.LGBMRegressor(**LGB_REG, objective="quantile", alpha=a).fit(tr[FD_FULL], tr[tg])
             for a in (0.05, 0.95)}
        s = np.maximum(q[0.05].predict(cal[FD_FULL]) - cal[tg].values,
                       cal[tg].values - q[0.95].predict(cal[FD_FULL]))
        n = len(s); qh = np.quantile(s, min(1.0, np.ceil((n + 1) * NOMINAL) / n))
        lo = np.clip(anchor * np.exp(q[0.05].predict(te[FD_FULL]) - qh) - 1, 0, None)
        hi = anchor * np.exp(q[0.95].predict(te[FD_FULL]) + qh) - 1
        y = te[tl].values
        fwd_rows.append({"horizon_weeks": h, "n_unit_weeks": len(te),
                         "weeks_covered": int(te.epi_week.nunique()),
                         "MAE_anchored": round(mean_absolute_error(y, pred), 3),
                         "MAE_persistence": round(mean_absolute_error(y, te.cases_lag0), 3),
                         "skill_vs_persistence_pct": round(100 * (1 - mean_absolute_error(y, pred) /
                                                                  mean_absolute_error(y, te.cases_lag0)), 2),
                         "conformal_coverage": round(float(((y >= lo) & (y <= hi)).mean()), 4)})
fwd = save_table("table10_forward_test_2026", pd.DataFrame(fwd_rows),
                 "True forward test: model frozen at end-2025, applied to 2026")
if len(fwd):
    print(fwd.to_string(index=False))
else:
    log.warning("no 2026 rows available — forward test skipped")


In [ ]:
# =============================================================================
# CELL 14 — Publication figures (300 dpi, IEEE single-column, CVD-safe)
# =============================================================================
plt.rcParams.update({
    "font.family": "serif", "font.size": 8, "axes.labelsize": 8.5,
    "axes.titlesize": 9, "legend.fontsize": 7.5, "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5, "figure.dpi": 300, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.4,
    "axes.axisbelow": True, "lines.linewidth": 1.4, "lines.markersize": 4,
})
COL1, COL2 = 3.45, 7.16       # IEEE single / double column width, inches

def _save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIGURES, f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig)
    log.info("figure -> %s.{png,pdf}", name)

# --- F1: national weekly series, folds shaded, data gap marked ----------------
nat = PANEL_D.groupby("week_start")["cases"].sum().reset_index()
fig, ax = plt.subplots(figsize=(COL2, 2.3))
ax.plot(nat.week_start, nat.cases, color=PAL["blue"], lw=1.2)
ax.fill_between(nat.week_start, 0, nat.cases, color=PAL["blue"], alpha=0.12)
for ty, c in zip(TEST_YEARS, [PAL["verm"], PAL["green"]]):
    ax.axvspan(pd.Timestamp(f"{ty}-01-01"), pd.Timestamp(f"{ty}-12-31"), color=c, alpha=0.10)
    ax.text(pd.Timestamp(f"{ty}-07-01"), nat.cases.max() * 0.94, f"test {ty}",
            ha="center", fontsize=7, color=c)
ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2021-12-31"), color="0.85", alpha=0.6)
ax.text(pd.Timestamp("2021-01-01"), nat.cases.max() * 0.55, "no district-wise\nreporting",
        ha="center", fontsize=7, color="0.35")
ax.set_ylabel("Reported cases per week"); ax.set_xlabel("")
ax.set_title("National weekly dengue incidence, 64 districts aggregated", loc="left")
_save(fig, "fig1_national_series_folds")

# --- F2: skill vs horizon, one axis, direct labels ---------------------------
fig, ax = plt.subplots(figsize=(COL2 * 0.62, 2.5))
series = [("level_L2", PAL["purple"], "o", "Level (L2)"),
          ("level_Tweedie", PAL["verm"], "s", "Level (Tweedie)"),
          ("anchored_L1", PAL["green"], "^", "Anchored growth (L1)")]
# Direct labels at the line ends; no legend box sitting over the data.
for key, c, mk, lab in series:
    v = [shootout[(shootout.horizon_weeks == h) & (shootout.key == key)]
         .skill_vs_persistence_pct.iloc[0] for h in HORIZONS]
    ax.plot(HORIZONS, v, color=c, marker=mk)
    ax.annotate(f"{lab}  {v[-1]:+.0f}%", (HORIZONS[-1], v[-1]), textcoords="offset points",
                xytext=(7, 0), fontsize=7, color=c, va="center", annotation_clip=False)
ax.axhline(0, color=PAL["grey"], lw=0.9, ls="--")
ax.annotate("persistence baseline", (1.0, 0), textcoords="offset points",
            xytext=(2, -9), fontsize=7, color=PAL["grey"])
ax.set_xticks(HORIZONS); ax.set_xlabel("Forecast horizon (weeks)")
ax.set_ylabel("MAE reduction vs persistence (%)")
ax.set_xlim(0.9, 4.15); ax.margins(y=0.18)
ax.set_title("Forecast skill by target parameterisation", loc="left")
_save(fig, "fig2_skill_by_horizon")

# --- F3: forecast vs actual with conformal band, two contrasting districts ----
I2 = INTERVALS[(2, "split")]
pick = [MD.groupby("unit")["cases"].mean().idxmax(),
        MD.groupby("unit")["cases"].mean().sort_values().index[len(MD.unit.unique()) // 4]]
fig, axes = plt.subplots(2, 1, figsize=(COL2, 3.6), sharex=True)
for ax, u in zip(axes, pick):
    g = I2[I2.unit == u].sort_values("week_start")
    ax.fill_between(g.week_start, g.lo, g.hi, color=PAL["blue"], alpha=0.20,
                    label="90% conformal interval", linewidth=0)
    ax.plot(g.week_start, g.med, color=PAL["blue"], lw=1.2, label="Forecast (median)")
    ax.plot(g.week_start, g.y, color=PAL["verm"], lw=1.0, ls="--", label="Observed")
    ax.set_ylabel("Cases"); ax.set_title(u, loc="left", fontsize=8.5)
axes[0].legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.02),
               handlelength=1.4, columnspacing=1.6)
axes[-1].set_xlabel("Test seasons 2024–2025")
fig.suptitle("Two-week-ahead forecasts with calibrated intervals", x=0.005, y=1.10,
             ha="left", fontsize=9)
_save(fig, "fig3_forecast_with_intervals")

# --- F4: coverage before and after calibration -------------------------------
fig, ax = plt.subplots(figsize=(COL2 * 0.62, 2.7))
xs = np.arange(len([1, 2, 4])); w = 0.26
for i, (key, c, lab) in enumerate([("raw", PAL["purple"], "Uncalibrated"),
                                   ("split", PAL["green"], "Split conformal"),
                                   ("adaptive", PAL["orange"], "Adaptive conformal")]):
    v = [calib[(calib.horizon_weeks == h) & (calib.key == key)].empirical_coverage.iloc[0]
         for h in [1, 2, 4]]
    ax.bar(xs + (i - 1) * w, v, w * 0.90, color=c, label=lab)
    for x, val in zip(xs + (i - 1) * w, v):
        ax.text(x, val + 0.014, f"{val:.2f}", ha="center", fontsize=6.5, color=c)
ax.axhline(NOMINAL, color=PAL["grey"], lw=1.0, ls="--")
ax.annotate("nominal 0.90", (2.62, NOMINAL), textcoords="offset points", xytext=(0, 3),
            fontsize=7, color=PAL["grey"], ha="left", annotation_clip=False)
ax.set_xticks(xs); ax.set_xticklabels([f"{h} wk" for h in [1, 2, 4]])
ax.set_xlim(-0.55, 2.6); ax.set_ylim(0.5, 1.0); ax.set_ylabel("Empirical coverage")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.16),
          handlelength=1.1, columnspacing=1.2)
ax.set_title("Interval coverage before and after calibration", loc="left")
_save(fig, "fig4_coverage_calibration")

# --- F5: optimism gap, ROC and PR as separate panels (never a dual axis) -----
m4 = matrix[matrix.condition != "OPTIMISM GAP (C1 - C4)"]
fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.4))
for ax, col, lab in zip(axes, ["ROC_AUC", "PR_AUC"], ["ROC-AUC", "PR-AUC"]):
    cols = [PAL["purple"], PAL["blue"], PAL["orange"], PAL["verm"]]
    ax.bar(range(len(m4)), m4[col], 0.62, color=cols)
    for i, v in enumerate(m4[col]):
        ax.text(i, v + 0.012, f"{v:.3f}", ha="center", fontsize=7)
    ax.set_xticks(range(len(m4)))
    ax.set_xticklabels(["C1\nrandom", "C2\ntime", "C3\nspace", "C4\nboth"])
    ax.set_ylim(0, 1.09); ax.set_ylabel(lab)
axes[0].set_title("Discrimination collapses under honest validation", loc="left")
_save(fig, "fig5_optimism_gap")

# --- F6: alarm lead time -----------------------------------------------------
if len(LEAD):
    fig, ax = plt.subplots(figsize=(COL1, 2.3))
    ax.hist(LEAD.lead_weeks_to_onset, bins=range(0, int(LEAD.lead_weeks_to_onset.max()) + 2),
            color=PAL["green"], alpha=0.85, edgecolor="white", linewidth=0.6)
    med = LEAD.lead_weeks_to_onset.median()
    ax.axvline(med, color=PAL["verm"], lw=1.3)
    ax.text(med + 0.25, ax.get_ylim()[1] * 0.88, f"median {med:.0f} wk",
            fontsize=7.5, color=PAL["verm"])
    ax.set_xlabel("Weeks of warning before the district crosses its outbreak threshold")
    ax.set_ylabel("District-seasons")
    ax.set_title("Operational lead time at 80% sensitivity", loc="left")
    _save(fig, "fig6_alarm_lead_time")


In [ ]:
# =============================================================================
# CELL 15 — Final gate and run manifest
# =============================================================================
cov2 = {h: (calib[(calib.horizon_weeks == h) & (calib.key == "raw")].empirical_coverage.iloc[0],
            calib[(calib.horizon_weeks == h) & (calib.key == "split")].empirical_coverage.iloc[0])
        for h in [1, 2, 4]}
for h, (raw, conf) in cov2.items():
    assert conf > raw, f"h={h}: conformal correction was a no-op"
    assert abs(conf - NOMINAL) < 0.06, f"h={h}: coverage {conf:.3f} is far from nominal"
log.info("[gate] calibration OK: %s", {h: round(c, 3) for h, (_, c) in cov2.items()})

# Every constant in this notebook, with how it was fixed. Anything marked
# "assumption" is a judgement call the paper must state rather than bury.
ASSUMPTIONS = pd.DataFrame([
    ("horizons", str(HORIZONS), "operational choice",
     "1-4 weeks is the window in which a health directorate can act"),
    ("test years", str(TEST_YEARS), "empirical",
     f"2026 excluded: {int((MD.year == 2026).sum())} rows over "
     f"{int(MD[MD.year == 2026].epi_week.nunique())} pre-monsoon weeks, not a full season; "
     "used instead as a frozen forward test in Cell 13"),
    ("selection split", f"train<={SEL_TRAIN}, validate {SEL_VALID}", "design",
     "no test year participates in any tuning decision"),
    ("tree hyperparameters", str({k: LGB_REG[k] for k in ('num_leaves', 'min_child_samples')}),
     "SELECTED", "inner-validation MAE sweep, Cell 4b / table1d"),
    ("tweedie_variance_power", f"{TWEEDIE_P}", "SELECTED", "inner-validation MAE sweep over 1.1-1.9"),
    ("growth objective", GROWTH_OBJ_NAME, "SELECTED", "inner-validation MAE, L1 vs L2"),
    ("conformal calibration weeks", str(CAL_WEEKS_BY_H), "SELECTED",
     "inner-validation |coverage - nominal| over 8/13/26/39, chosen separately per horizon"),
    ("nominal coverage", f"{NOMINAL}", "reporting convention", "90% is standard for epidemic forecast hubs"),
    ("significance level", f"{ALPHA}", "convention",
     "0.05 two-sided; every family of tests is Benjamini-Hochberg corrected"),
    ("alarm quantile", f"{ALARM_QUANTILE}", "assumption",
     f"per-unit 80th percentile of training-year cases; base rate {A[YA].mean():.3f}. "
     "Sensitivity across 0.70/0.75/0.80/0.85 is not run - state as a limitation"),
    ("alarm minimum cases", f"{ALARM_MIN_CASES}", "assumption",
     "floor so that near-zero districts cannot alarm on a single case"),
    ("case lags", "[1,2,3,4,8] weeks", "assumption",
     "covers the intrinsic + extrinsic incubation cycle; not swept"),
    ("climate lags", "[2,3,4] weeks", "assumption",
     "vector-development lag from the entomological literature; not swept"),
    ("modelling window", "year >= 2022", "empirical",
     "2019 covers only 30% of its season (30,257 of 101,354 official) and 2020-21 are absent"),
    ("reconciliation tolerance", str(RECON_TOL_PCT), "POST-HOC",
     "the divisional bound was widened to 2.5% after observing a -1.85% gap in 2022; "
     "this is a post-hoc choice and is reported as such"),
    ("burden tertiles", "3 equal-count groups", "descriptive", "cut on training-year mean cases only"),
    ("missing-report zeros", f"{len(gap_key)} weeks flagged, retained", "empirical",
     "cases=0 with patients still admitted; Cell 12b shows whether removing them changes the result"),
    ("rainfall source", "district CHIRPS vs divisional NASA POWER", "KNOWN MISMATCH",
     "r=0.86 between products (table1c); Cell 12b re-runs the resolution test without rain"),
    ("bootstrap blocks", "(unit, fold)", "design",
     "resampling whole district-seasons preserves the serial correlation that a row-level "
     "bootstrap would destroy"),
], columns=["constant", "value", "basis", "evidence"])
save_table("table0_assumptions_register", ASSUMPTIONS,
           "Every constant, and whether it was selected empirically or assumed")
print(ASSUMPTIONS.to_string(index=False))

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED, "test_years": TEST_YEARS, "horizons": HORIZONS,
    "nominal_coverage": NOMINAL, "calibration_weeks": CAL_WEEKS_BY_H,
    "alpha": ALPHA, "selected_hyperparameters": LGB_REG,
    "selected_tweedie_power": TWEEDIE_P, "selected_growth_objective": GROWTH_OBJ_NAME,
    "selection_split": {"train_max_year": SEL_TRAIN, "validate_year": SEL_VALID},
    "district_rows": int(len(MD)), "district_units": int(MD.unit.nunique()),
    "district_features_full": len(FD_FULL), "district_features_shared": len(FD_SHR),
    "divisional_rows": int(len(MV)) if MV is not None else 0,
    "forbidden_columns_blocked": sorted(FORBIDDEN),
    "tables": _TABLES,
    "figures": sorted(os.path.basename(p) for p in glob.glob(os.path.join(FIGURES, "*.png"))),
}
with open(os.path.join(RESULTS, "run_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)

log.info("=" * 78)
log.info("RUN COMPLETE — %d tables, %d figures", len(_TABLES), len(manifest["figures"]))
for k, v in _TABLES.items():
    log.info("  %-34s %3d rows  %s", k, v["rows"], v["caption"])
log.info("=" * 78)
